<a href="https://www.kaggle.com/code/nadijfer/spam-email-using-ann-from-scratch?scriptVersionId=288839977" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# **Detecting Spam Email using Simple Neural Network from Scratch**

# I. Introduction

This notebook is my attempt on implementing what I've learnt in class, for have taken natural language processing (NLP) and artificial neural network (ANN). It aims to test my understanding both mathematically and in code implementation by building from scratch without any modules, except modules below. References primarily used from lecture notes and class' presentations. For this notebook, I tried to write codes and its description as clear as possible, by adding docstrings to functions.

The notebook will be separated into two implementation: using vanilla Bag-of-Words (BoW) for baseline, TF-IDF and  Word Embeddings as its improvement.

# Baseline Implementation using BoW

In [1]:
import numpy as np
import pandas as pd
import string
import time

# II. Exploratory Data Analysis (EDA)

In [2]:
df = pd.read_csv('/kaggle/input/spam-email-dataset/emails.csv')
df

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1
...,...,...
5723,Subject: re : research and development charges...,0
5724,"Subject: re : receipts from visit jim , than...",0
5725,Subject: re : enron case study update wow ! a...,0
5726,"Subject: re : interest david , please , call...",0


### Null or NaN values

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5728 entries, 0 to 5727
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    5728 non-null   object
 1   spam    5728 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 89.6+ KB


No null or NaN value with equal distribution of both classes. We're good to go.

# II. Text Processing

## Cleaning and Tokenizing Documents

In [4]:
def preprotext(text):
    """Cleaning text input by applying lowercase, removing punctuation, and tokenizing (text.split()). The function could be used as:

    cleaned_texts = texts.copy()
    for i in range(len(texts)):
        cleaned_texts[i] = preprotext(texts[i])

    Args:
        text: string to be cleaned.

    Returns:
        list: tokens of cleaned text.
    """
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation)) # remove punctuation marks
    tokens = text.split() # tokenizing
    return tokens

## Building Vocabulary

Vocabulary $|V|$ is a list of words model used as dictionary.

In [5]:
def build_vocab(documents):
    """Building list of words from train data for model. The function could be used as:
    vocab = build_vocab(cleaned_texts)
    len(vocab), vocab

    Args:
        documents: list of words

    Returns:
        list: vocabulary
    """
    vocab = []
    for document in documents:
        tokens = preprotext(document) # resulting tokens of each documents
        for token in tokens:
            vocab.append(token)
    
    vocab = list(set(vocab))
    vocab = {word: index for index, word in enumerate(vocab)}
    return vocab

## Encoding Text with Bag-of-Words

In [6]:
def build_bow(cleaned_texts, vocab):
    """Build BoW to a token of clened texts
    doc_bow = [] # bow for each documents
    for docs in cleaned_texts:
        docs = build_bow(docs)
        doc_bow.append(docs)
    """
    bow = np.zeros((len(vocab)))
    for token in cleaned_texts:
        if token in vocab:
            bow[vocab[token]] += 1
    return bow

# IV. Training Neural Network

The architecture of NN is built upon Multilayer Perceptron (MLP) with a hidden layer consisting 128 hidden nodes $H$. For binary target $t$, the loss function $\mathcal L$ used is binary cross entropy (BCE), defined as:

$$
\mathcal{L} = - \sum y \cdot \log (\hat y)
$$

Where $y$ is the true label (we defined in this notebook as `t` for target) and $\hat{y}$ is the prediction label (defined as `y`). 

Activation functions for input-to-hidden layer is ReLU, the function itself and its derivative defined as:

$$
\begin{align*}
f(x) & = \mathrm{ReLU} = \begin{cases}x, & \text{if } x > 0\\ 0, & \text{otherwise}\end{cases}\\
f'(x) & = \begin{cases}1\\ 0\end{cases}\\
\end{align*}
$$

and the activation functions for hidden-to-output layer is Sigmoid, both the function itself and its derivative defined as:

$$
\begin{align*}
f(x) & = \sigma = \frac{1}{1 + e^{-x}} \\
f'(x) & = f(x) (1-f(x))
\end{align*}
$$

Such architecture could be implemented as:

```
Input(len(vocab)) -> Hidden(128; Relu -> Sigmoid) -> y(1) -> L(BCE)
```

In [7]:
N = len(df) # length of the training data
H = 128 # hidden nodes (size), 1 hidden layer
lr = 1e-4 # learning rate

### Activation functions and its derivatives

In [8]:
def relu(x):
    return np.maximum(0, x)

def relu_deriv(x):
    return (x > 0).astype(float)

def sigmoid(x):
    return 1/(1+np.exp(-x))

def sigmoid_deriv(x):
    return sigmoid(x) * (1-sigmoid(x))

### Loss function and its derivative (BCE)

In [9]:
def BCE(t, y):
    return - (t * np.log(y))

def BCE_deriv(t, y): # only when sigmoid is used
    return y - t

## Train-test split

In [10]:
X = df["text"]
y = df["spam"]

idx = np.random.permutation(N)

split = int(0.8 * N)

X_train = X[:split]
y_train = y[:split]

X_test = X[split:]
y_test = y[split:]

## Building Vocabulary, Cleaning Text and Encoding BoW

In [11]:
vocab = build_vocab(X_train)

In [12]:
train_cleaned = X_train.copy()
for i in range(len(X_train)):
    train_cleaned.iloc[i] = preprotext(train_cleaned.iloc[i])

In [13]:
train_bow = []
for tokens in train_cleaned:
    tokens = build_bow(tokens, vocab)
    train_bow.append(tokens)

## Initializing NN parameters

In [14]:
x = np.array(train_bow) # input
t = y_train # target
w = np.random.randn(x.shape[1], H) * np.sqrt(2/len(X_train)) # He initialization
b0 = np.zeros(H) # hidden bias
v = np.random.randn(H, 1) * np.sqrt(1/H) # Xavier initialization
b1 = np.zeros(1) # output bias

In [15]:
# this cell is to check the values before and after training
params = {
    'w': w,
    'b0': b0,
    'v': v,
    'b1': b1
}

params

{'w': array([[-0.00326743,  0.00610691, -0.01049602, ...,  0.00480147,
         -0.01121581,  0.00438979],
        [-0.00273346, -0.00258601, -0.00584178, ...,  0.00812692,
          0.00244507, -0.01139967],
        [ 0.01904576, -0.02340002, -0.02764868, ..., -0.00082169,
         -0.03702105, -0.00992453],
        ...,
        [-0.00374429, -0.02329137,  0.02222826, ...,  0.00343197,
         -0.02078372,  0.03245609],
        [ 0.01651059, -0.00542069,  0.02076001, ..., -0.0213649 ,
          0.00701634, -0.00532796],
        [ 0.01999225,  0.00052684,  0.01903666, ...,  0.00994869,
          0.00070451, -0.01554271]]),
 'b0': array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.

In [16]:
x.shape, w.shape, b0.shape, v.shape, b1.shape

((4582, 34609), (34609, 128), (128,), (128, 1), (1,))

## Backpropagation

Backpropagation to a parameter at input layer $\theta$ could be defined as:

$$
\frac{\partial L}{\partial \theta} = \frac{\partial L}{\partial y_{out}} \frac{\partial y_{out}}{\partial y_{in}} \frac{\partial y_{in}}{\partial z_{out}} \frac{\partial z_{out}}{\partial z_{in}} \frac{\partial z_{in}}{\partial \theta}
$$

Special condition occurs to binary cross entropy (BCE) loss function when is used next to sigmoid activation function. Both BCE and its derivative could be defined as:

$$
\begin{align}
f(x) = \mathcal{L} &= - \sum^N_{i=1} y \cdot \log(\hat{y}) \\
f'(x) &= \hat{y} - y
\end{align}
$$

Where $y$ is the true label (we defined in this notebook as `t` for target) and $\hat{y}$ is the prediction label (defined as `y`), the function could be simplified as:

$$
\begin{align}
\frac{\partial L}{\partial \theta} &= \frac{\partial L}{\partial y_{out}} \frac{\partial y_{out}}{\partial y_{in}} \frac{\partial y_{in}}{\partial z_{out}} \frac{\partial z_{out}}{\partial z_{in}} \frac{\partial z_{in}}{\partial \theta} \\
 & = (\hat{y}-y) \cdot v \cdot \mathrm{ReLU}'(z_{in}) \cdot \Omega
\end{align}
$$

Where $\Omega$ depends whether observing input biases or weight biases, such as:

$$
\Omega = 
    \begin{cases}
        x, & \text{if } \theta = w,\\
        1, & \text{if } \theta = b_0
    \end{cases}
$$

Gradient to parameter $\theta$ at hidden layer could be defined as:
$$
\begin{align}
\frac{\partial L}{\partial \theta} &= \frac{\partial L}{\partial y_{out}} \frac{\partial y_{out}}{\partial y_{in}} \frac{ \partial y_{in} }{ \partial \theta }  \\
 & = (\hat{y}-y) \cdot \Omega
\end{align}
$$

And again, $\Omega$ depends whether observing input biases or weight biases, such as:

$$
\Omega = 
    \begin{cases}
        z, & \text{if } \theta = v,\\
        1, & \text{if } \theta = b_1
    \end{cases}
$$

In [17]:
def backpropagation(x_i, y, t_i, z_in, z_out, w, b0, v, b1, lr):
    
    delta_out = BCE_deriv(t_i, y)
    
    dv = z_out.reshape(-1, 1) * delta_out
    db1 = delta_out

    delta_hidden = (v.flatten() * delta_out) * relu_deriv(z_in)
    
    dw = np.outer(x_i, delta_hidden)
    db0 = delta_hidden
        
    v  -= lr * dv
    b1 -= lr * db1
    w  -= lr * dw
    b0 -= lr * db0

    return w, b0, v, b1

## Training

In [18]:
max_epochs = 10

start = time.time()
for epoch in range(max_epochs):
    total_loss = 0
    
    for i in range(len(x)):
        z_in = x[i] @ w + b0
        z_out = relu(z_in)
        y_in = z_out @ v + b1
        y = sigmoid(y_in)

        loss = BCE(t[i], y)
        total_loss += loss

        w, b0, v, b1 = backpropagation(x[i], y, t[i], z_in, z_out, w, b0, v, b1, lr)

    print(f'average loss epoch {epoch+1}: {total_loss / len(x)}')
    
end = (time.time() - start) / 60
print(f'\ntraining time: {end:.2f} minutes')

average loss epoch 1: [0.14186487]
average loss epoch 2: [0.21204]
average loss epoch 3: [0.17033768]
average loss epoch 4: [0.14609449]
average loss epoch 5: [0.12828783]
average loss epoch 6: [0.11445781]
average loss epoch 7: [0.10519842]
average loss epoch 8: [0.09731883]
average loss epoch 9: [0.09132858]
average loss epoch 10: [0.08636585]

training time: 32.86 minutes


In [19]:
params

{'w': array([[-0.00326743,  0.00610691, -0.01051284, ...,  0.00480147,
         -0.01124748,  0.00440191],
        [-0.00269223, -0.00257194, -0.00571921, ...,  0.00817447,
          0.00270358, -0.01175334],
        [ 0.01902432, -0.02344214, -0.02763731, ..., -0.0008217 ,
         -0.0370206 , -0.00989974],
        ...,
        [-0.00378786, -0.02333489,  0.0221703 , ...,  0.00340275,
         -0.02090011,  0.03259933],
        [ 0.0165106 , -0.00542058,  0.02076006, ..., -0.0213649 ,
          0.00701644, -0.00532803],
        [ 0.01999553,  0.00053789,  0.01905004, ...,  0.00994869,
          0.00073177, -0.01555295]]),
 'b0': array([ 2.07769911e-03,  6.28267887e-04,  2.69091160e-03,  4.58302684e-04,
         8.37807075e-04,  3.44816436e-03,  4.64609171e-06, -2.62343528e-05,
        -9.42882353e-05,  7.13203527e-04,  1.31578423e-02,  1.57642910e-03,
         1.71277622e-05,  1.22584539e-02,  7.42535288e-04,  9.39536604e-04,
         5.41962141e-03, -1.51593298e-06, -6.47240750e-04,

# V. Inference

## Cleaning Text and Encoding BoW for Data Test

In [20]:
test_cleaned = X_test.copy()
for i in range(len(X_test)):
    test_cleaned.iloc[i] = preprotext(test_cleaned.iloc[i])

In [21]:
test_bow = []
for tokens in test_cleaned:
    tokens = build_bow(tokens, vocab)
    test_bow.append(tokens)

## Prediction

In [22]:
xt = np.array(test_bow)
y_preds = []
    
for i in range(len(xt)):
    z_in = xt[i] @ w + b0
    z_out = relu(z_in)
    y_in = z_out @ v + b1
    y = sigmoid(y_in)
    y_preds.append(y)

y_preds = np.array(y_preds)

In [23]:
y_hat = (y_preds >= 0.5).astype(int)
accuracy = round((y_hat.flatten() == y_test).mean(), 4) * 100

In [24]:
accuracy

np.float64(99.91)

# VI. Discussion

Although the test accuracy reached 99.56%, I could say the model overfits and this accuracy can be concluded false. To observe tokens in data test appeared in vocabulary, we may use:

In [25]:
for i in range(len(xt)):
    if xt[0][i] != 0:
        print(i)

458
571
1010


From the results above, we can conclude that Vanilla BoW did not work well to the dataset. This is primarily because BoW splits words independently without context. Further improvement would be made with TF-IDF and Embeddings.

# Improvement with TF-IDF

In [26]:
"""TODO(): implement improvements with TF-IDF"""

'TODO(): implement improvements with TF-IDF'

# Improvement with Word Embeddings

In [27]:
"""TODO(): implement improvements with Word Embeddings"""

'TODO(): implement improvements with Word Embeddings'